# ClaudeAgentOptions Explained

`ClaudeAgentOptions` is the config object that controls how the agent behaves: which model answers, what system prompt frames its personality and constraints, which tools it can use, and more. This episode covers two of the most-used fields: `model` and `system_prompt`.


In [1]:
from claude_agent_sdk import (
    query,  # one-shot function: ask something, get a stream of messages back
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)


async def run_prompt(options: ClaudeAgentOptions, prompt: str) -> str:
    # options: the ClaudeAgentOptions settings to run this prompt with
    # prompt:  the question/task to send to Claude

    # ResultMessage is the last message in the stream — it carries the final
    # answer plus stats (cost, duration, etc). We only care about `.result` here.
    result = ""
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, ResultMessage):
            result = message.result or ""
    return result

## A custom `system_prompt` visibly changes the output style


In [2]:
# system_prompt: instructions that apply to every message in this session.
# Think of it as "the personality/rules Claude must always follow" —
# here we're telling it to keep answers short and to-the-point.
terse_options = ClaudeAgentOptions(
    model="haiku",
    system_prompt="You are a terse assistant, answer in one sentence only.",
)


async def demo_system_prompt() -> None:
    answer = await run_prompt(terse_options, "What causes seasons on Earth?")
    print(f"Custom system_prompt (terse, one sentence):\n{answer}")


await demo_system_prompt()

Custom system_prompt (terse, one sentence):
Earth's tilted axis (about 23.5° from vertical) causes different hemispheres to receive varying amounts of direct sunlight as the planet orbits the sun, creating seasons.


## Switching models is just a different `ClaudeAgentOptions`


In [3]:
# model: which Claude model answers your prompt.
# "haiku"  -> fastest and cheapest, good for simple tasks
# "sonnet" -> smarter and more capable, good for harder tasks
# Everything else about the two options below is identical — only the model differs.
haiku_options = ClaudeAgentOptions(model="haiku")
sonnet_options = ClaudeAgentOptions(model="sonnet")


async def demo_model_switch() -> None:
    prompt = "In two sentences, explain what a black hole is."
    haiku_answer = await run_prompt(haiku_options, prompt)
    sonnet_answer = await run_prompt(sonnet_options, prompt)
    print(f"model='haiku':\n{haiku_answer}\n")
    print(f"model='sonnet':\n{sonnet_answer}")


await demo_model_switch()

model='haiku':
A black hole is a region of spacetime where gravity is so intense that nothing, not even light, can escape from beyond its event horizon (the point of no return). Black holes form when massive stars collapse at the end of their lives, compressing all their matter into an infinitely dense point called a singularity.

model='sonnet':
A black hole is a region of space where gravity is so intense that nothing—not even light—can escape once it crosses the boundary known as the event horizon. It typically forms when a massive star collapses under its own gravity at the end of its life, compressing enormous mass into an extremely small volume.


## Summary

- `system_prompt` reshapes how Claude responds without touching your prompts.
- `model` swaps which model answers — both are just fields on the same `ClaudeAgentOptions` object.
